# NEXUS-AURORA: Ablation Studies

Runs structured ablations to validate each architectural component:

| Day | Ablation | Purpose |
|-----|----------|---------|
| 4 | LLaMA baseline vs S-only | Baseline comparison |
| 5 | S+R K=1 vs adaptive K | Value of adaptive iteration |
| 6 | S+V only vs full AURORA | Value of reasoning stream |
| 7 | Bridge top_k (1,4,8,16) | Optimal sparse attention |
| 8 | K_max (2,4,6) | Optimal reasoning depth |
| 9 | n_slots (16,32,64) | Optimal slot count |
| 10 | Muon vs AdamW | Optimizer comparison |

Each ablation trains for 200M tokens. Results measured in BPB (bits per byte).

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/nexus-lm')

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

In [ ]:
# Paths
TRAIN_BIN  = '/kaggle/input/nexus-data/fineweb_edu_train.bin'
VAL_BIN    = '/kaggle/input/nexus-data/fineweb_edu_val.bin'
CONFIG     = '/kaggle/working/nexus-lm/config/nexus_aurora_v1.yaml'
OUTPUT_DIR = '/kaggle/working/ablation_results'

In [ ]:
# Run Day 4 ablations: baseline vs S-only
from evaluation.ablation_runner import run_ablation, ABLATION_CONFIGS
import yaml

with open(CONFIG) as f:
    base_config = yaml.safe_load(f)

day4_ablations = ['baseline', 'aurora_s_only']
results = []

for name in day4_ablations:
    result = run_ablation(
        name, base_config, ABLATION_CONFIGS[name],
        TRAIN_BIN, VAL_BIN, OUTPUT_DIR,
        max_tokens=200_000_000,
    )
    results.append(result)
    print(f'{name}: BPB={result["bpb"]:.4f}')

In [ ]:
# Plot results
import matplotlib.pyplot as plt

names = [r['name'] for r in results]
bpbs  = [r['bpb']  for r in results]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, bpbs, color=['#e74c3c' if n == 'baseline' else '#3498db' for n in names])
plt.ylabel('BPB (bits per byte) — lower is better')
plt.title('NEXUS-AURORA Ablation Results')
for bar, bpb in zip(bars, bpbs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{bpb:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/ablation_results.png', dpi=150)
plt.show()